# 28 – PydanticAI Structured Agents

## Learning objectives

- Declare a `pydantic_ai.Agent` with a Pydantic `output_type`.
- Inject runtime dependencies via a deps type and `RunContext`.
- Understand how strict schemas enable self-correction loops (`retries`).

## About this topic

PydanticAI combines LLM calls with Pydantic validation: the model must produce data that parses into your output type. Dependency injection keeps prompts and tools free of global state. When validation fails, the library can retry with error feedback—useful for brittle structured outputs.

In [ ]:
from pathlib import Path
import importlib.util

def _repo_root() -> Path:
    cwd = Path.cwd().resolve()
    return cwd if (cwd / "pyproject.toml").exists() else cwd.parent

path = _repo_root() / "examples/library/pydantic_ai_agents/structured_agent.py"
spec = importlib.util.spec_from_file_location("pai_struct", path)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
mod.demo_agent_definition()

In [ ]:
from dataclasses import dataclass
from pydantic import BaseModel, Field

# Minimal pattern mirroring the example: deps + output schema without calling the network
@dataclass
class SearchDeps:
    corpus_id: str = "default"

class Answer(BaseModel):
    summary: str = Field(min_length=5)
    confidence: float = Field(ge=0, le=1)

try:
    from pydantic_ai import Agent
    agent: Agent[SearchDeps, Answer] = Agent(
        "openai:gpt-4o-mini",
        output_type=Answer,
        deps_type=SearchDeps,
        system_prompt="Answer briefly using corpus {deps.corpus_id}.",
    )
    print("Agent ready:", agent)
    print("Output JSON Schema title:", Answer.model_json_schema().get("title"))
except ImportError:
    print("Install: pip install pydantic-ai")
    print("Schema preview:", Answer.model_json_schema().get("properties", {}).keys())

In [ ]:
from pathlib import Path
import importlib.util

def _repo_root() -> Path:
    cwd = Path.cwd().resolve()
    return cwd if (cwd / "pyproject.toml").exists() else cwd.parent

path = _repo_root() / "examples/library/pydantic_ai_agents/self_correction.py"
spec = importlib.util.spec_from_file_location("pai_sc", path)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
mod.demo_self_correction()

## Exercises and next steps

1. Add a tool function to a PydanticAI agent and print the tool JSON Schema bundle your provider expects.
2. Lower `retries` to 1 in the self-correction example and observe behavior when the mock LLM returns invalid JSON (write a small test double).
3. Compare PydanticAI deps with FastAPI `Depends`—when would you bridge them in an API route?